In [0]:
from pyspark.sql.functions import *

gold_skill_demand_realtime = (
    spark.table("skillpulse.silver.silver_realtime_skills")
    .groupBy("skill")
    .agg(
        countDistinct("job_id").alias("job_count")
    )
)

(
    gold_skill_demand_realtime.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "skillpulse.gold.gold_skill_demand_realtime"
    )
)

In [0]:
gold_role_skill_mapping_realtime = (
    spark.table("skillpulse.silver.silver_realtime_skills")
    .groupBy(
        "normalized_role",
        "skill"
    )
    .agg(
        countDistinct("job_id").alias("skill_demand")
    )
)

(
    gold_role_skill_mapping_realtime.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "skillpulse.gold.gold_role_skill_mapping_realtime"
    )
)

In [0]:
gold_country_skill_mapping_realtime = (
    spark.table("skillpulse.silver.silver_realtime_skills")
    .join(
        spark.table("skillpulse.silver.silver_jobs_realtime")
            .select("job_id", "country"),
        "job_id"
    )
    .groupBy(
        "country",
        "skill"
    )
    .agg(
        countDistinct("job_id").alias("skill_demand")
    )
)

(
    gold_country_skill_mapping_realtime.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "skillpulse.gold.gold_country_skill_mapping_realtime"
    )
)

In [0]:
from pyspark.sql.functions import *

snapshot_df = (
    spark.table(
        "skillpulse.gold.gold_skill_demand_realtime"
    )
    .withColumn(
        "snapshot_date",
        current_date()
    )
    .withColumn(
        "snapshot_timestamp",
        current_timestamp()
    )
)

(
    snapshot_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(
        "skillpulse.gold.gold_skill_daily_snapshot"
    )
)

print(
    "Snapshot Rows Saved:",
    snapshot_df.count()
)

Snapshot Rows Saved: 37


In [0]:
from pyspark.sql.functions import *

role_demand = (
    spark.table("skillpulse.silver.silver_jobs")
    .groupBy("job_title")
    .agg(count("*").alias("job_count"))
    .orderBy(desc("job_count"))
)

(
    role_demand.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "skillpulse.gold.gold_role_demand_realtime"
    )
)

print("Role Demand Rows:", role_demand.count())

Role Demand Rows: 6485


In [0]:
from pyspark.sql.functions import *

country_demand = (
    spark.table("skillpulse.silver.silver_jobs")
    .groupBy("search_country")
    .agg(count("*").alias("job_count"))
    .withColumnRenamed(
        "search_country",
        "country"
    )
)

(
    country_demand.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "skillpulse.gold.gold_country_demand_realtime"
    )
)

print("Country Demand Rows:", country_demand.count())

Country Demand Rows: 5


In [0]:
from pyspark.sql.functions import *

exclude = [
    "Jobs for Humanity",
    "Dice",
    "Recruiting from Scratch",
    "ClearanceJobs"
]

companies = (
    spark.table("skillpulse.silver.silver_jobs")
    .filter(~col("company").isin(exclude))
    .groupBy("company")
    .agg(count("*").alias("job_count"))
    .orderBy(desc("job_count"))
)

(
    companies.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(
        "skillpulse.gold.gold_top_companies_realtime"
    )
)

In [0]:
from pyspark.sql.functions import *

skill_trends = (
    spark.table(
        "skillpulse.gold.gold_skill_daily_snapshot"
    )
    .groupBy(
        "snapshot_date",
        "skill"
    )
    .agg(
        sum("job_count").alias("job_count")
    )
)

(
    skill_trends.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "skillpulse.gold.gold_skill_trends"
    )
)

In [0]:
from pyspark.sql.functions import *

skill_role_demand = (
    spark.table("skillpulse.silver.silver_job_skill_enriched_v2")
    .groupBy("job_title", "skill")
    .agg(
        countDistinct("job_link").alias("job_count")
    )
)

(
    skill_role_demand.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "skillpulse.gold.gold_skill_role_demand"
    )
)

print(
    "gold_skill_role_demand rows:",
    skill_role_demand.count()
)

gold_skill_role_demand rows: 232301


In [0]:
from pyspark.sql.functions import *

company_skill_demand = (
    spark.table("skillpulse.silver.silver_job_skill_enriched_v2")
    .groupBy("company", "skill")
    .agg(
        countDistinct("job_link").alias("job_count")
    )
)

(
    company_skill_demand.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "skillpulse.gold.gold_company_skill_demand"
    )
)

print(
    "gold_company_skill_demand rows:",
    company_skill_demand.count()
)

gold_company_skill_demand rows: 219205


In [0]:
bronze_jobs = spark.table(
    "skillpulse.bronze.bronze_jobs_realtime"
).count()

silver_jobs = spark.table(
    "skillpulse.silver.silver_jobs_realtime"
).count()

extracted_skills = spark.table(
    "skillpulse.silver.silver_realtime_skills"
).count()

unique_skills = (
    spark.table("skillpulse.silver.silver_realtime_skills")
    .select("skill")
    .distinct()
    .count()
)

print("bronze_jobs =", bronze_jobs)
print("silver_jobs =", silver_jobs)
print("extracted_skills =", extracted_skills)
print("unique_skills =", unique_skills)

bronze_jobs = 1859
silver_jobs = 1859
extracted_skills = 1094
unique_skills = 38


In [0]:
from pyspark.sql.functions import current_timestamp

pipeline_health = spark.createDataFrame(
    [(
        bronze_jobs,
        silver_jobs,
        extracted_skills,
        unique_skills
    )],
    [
        "bronze_job_count",
        "silver_job_count",
        "extracted_skills",
        "unique_skills"
    ]
)

from pyspark.sql.functions import *

pipeline_health = pipeline_health.withColumn(
    "last_run_time",
    from_utc_timestamp(
        current_timestamp(),
        "Asia/Kolkata"
    )
)

(
    pipeline_health.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "skillpulse.gold.gold_pipeline_health"
    )
)

print("Pipeline health updated")

Pipeline health updated


In [0]:
from pyspark.sql.functions import col, to_date

today = spark.sql("SELECT current_date()").collect()[0][0]

spark.sql(f"""
DELETE FROM skillpulse.monitoring.pipeline_run_history
WHERE DATE(snapshot_timestamp) = '{today}'
""")

(
    spark.table("skillpulse.gold.gold_pipeline_health")
    .selectExpr(
        "last_run_time as snapshot_timestamp",
        "bronze_job_count",
        "silver_job_count",
        "extracted_skills",
        "unique_skills"
    )
    .write
    .mode("append")
    .saveAsTable("skillpulse.monitoring.pipeline_run_history")
)

In [0]:
%sql
SELECT *
FROM skillpulse.monitoring.pipeline_run_history
ORDER BY snapshot_timestamp;

snapshot_timestamp,bronze_job_count,silver_job_count,extracted_skills,unique_skills
2026-06-09T10:16:39.987Z,1872,1872,1068,37
2026-06-10T15:24:14.551Z,1856,1856,1117,38
2026-06-11T14:33:29.466Z,1859,1859,1094,38
